# Generación de Batería de Preguntas para TFG

Este notebook tiene dos partes:
1. **Exploración de la Base de Datos:** Usamos Elasticsearch con parámetros flexibles para descubrir de qué hablan las noticias indexadas y extraer hechos concretos.
2. **Generación del Dataset:** Creación y guardado del archivo `bateria_preguntas_tfg.csv` con el formato requerido por el Juez LLM.

In [2]:
from elasticsearch import Elasticsearch

# 1. CONEXIÓN (Túnel SSH)
es = Elasticsearch("http://127.0.0.1:9250")

try:
    if es.ping():
        print(f"Conectado a: {es.info()['name']} (Listo para explorar)")
    else:
        print("El servidor no responde. ¿Está el túnel abierto?")
except Exception as e:
    print(f"Error inesperado: {e}")

# 2. FUNCIÓN DE EXPLORACIÓN (Relajada)
def explorar_noticias(query, k=4): # k=3 para no saturar la pantalla
    print(f"\nEXPLORANDO TEMA: '{query}'")
    print("=" * 80)
    
    search_payload = {
        "size": k,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^3", "body"], 
                "fuzziness": "AUTO", # Tolerancia a errores (solo para explorar)
                "operator": "or"     # Búsqueda amplia
            }
        },
        # IMPORTANTE: Traemos el body para poder leer el dato exacto
        "_source": ["title", "date", "source", "body"] 
    }
    
    try:
        response = es.search(index="noticias_tfg", body=search_payload) 
        hits = response['hits']['hits']
        
        if not hits:
            print("No se encontraron noticias sobre este tema.")
            return

        for i, hit in enumerate(hits):
            data = hit['_source'] 
            clean_date = data.get('date', 'N/A')[:10] if data.get('date') else "Sin fecha"
            
            print(f"{i+1}. TÍTULO: {data.get('title', 'Sin título')}")
            print(f"FECHA:  {clean_date} | FUENTE: {data.get('source', 'Desconocida')}")
            # Imprimimos los primeros 400 caracteres del cuerpo para buscar el "dato"
            cuerpo = data.get('body', '')
            print(f"TEXTO:  {cuerpo}...\n")
            print("-" * 80)
            
    except Exception as e:
        print(f"Error durante la búsqueda: {e}")

Conectado a: valencia (Listo para explorar)


In [3]:
# CASO 1: Ciencia y Espacio (Suele tener fechas y nombres de misiones)
explorar_noticias("mision espacial NASA SpaceX Luna Marte")

# CASO 2: Salud y Medicina (Avances, vacunas, enfermedades)
explorar_noticias("avance medico tratamiento cancer vacuna")

# CASO 3: Deportes alternativos (Tenis o Motor)
explorar_noticias("Carlos Alcaraz tenis Grand Slam") 
# o también: explorar_noticias("Fernando Alonso Formula 1 Aston Martin")

# CASO 4: Economía Doméstica y Criptomonedas
explorar_noticias("precio vivienda alquiler España inflación")
# o también: explorar_noticias("Bitcoin criptomonedas récord histórico")

# CASO 5: Política Internacional (Conflictos o elecciones en otros continentes)
explorar_noticias("elecciones America Latina presidente")
# o también: explorar_noticias("conflicto Israel Gaza Oriente Medio")

# CASO 6: Tecnología de Consumo / Redes Sociales
explorar_noticias("prohibición TikTok Estados Unidos redes sociales")


EXPLORANDO TEMA: 'mision espacial NASA SpaceX Luna Marte'
1. TÍTULO: Guerra en la NASA por su jefatura y por el retraso de SpaceX en la misión a la Luna
FECHA:  2025-10-22 | FUENTE: La Vanguardia
TEXTO:  En los despachos de la NASA sonaría que ni pintada una canción de Pink Floyd en estas ajetreadas jornadas. “En realidad, no hay un lado oscuro de la luna. De hecho, todo es oscuro”, entonaron en su tema Eclipse . La agencia espacial estadounidense se halla también en la oscuridad, inmersa en problemas muy terrestres que amenazan el proyecto de volver a poner los pies en el satélite de la tierra, previsto para el 2027 con la llamada misión Artemis. La NASA carece, desde la llegada de Donald Trump a la Casa Blanca, de un administrador y todo apunta que ha surgido una lucha de poder que no hace más que añadir incertidumbre a lo proyectos de futuro y no solo en el terreno de las galaxias. Este contencioso se ha desatado entre el administrador interino, Sean Duffy, actual secretario de Tra

In [4]:
explorar_noticias("¿Cuánto dinero perdió Elon Musk tras el anuncio de los aranceles?")


EXPLORANDO TEMA: '¿Cuánto dinero perdió Elon Musk tras el anuncio de los aranceles?'


1. TÍTULO: Esta es la fortuna que ha perdido Elon Musk desde que Trump anunció sus aranceles
FECHA:  2025-04-07 | FUENTE: La Razón
TEXTO:  Manifestaciones, ataques contra sus locales y vehículos eléctricos y hundimiento de las ventas. El apoyo de Elon Musk al Gobierno de Donald Trump le está saliendo demasiado caro. 17.800 millones de dólares, en concreto, según los cálculos de XTB. Esta es la fortuna que perdió en solo dos jornadas por el batacazo bursátil de Tesla el jueves y viernes pasado después de que Trump anunciase su paquete arancelario contra el mundo entero en el "Día de la Liberación". La compañía de vehículos eléctricos del magnate se despeñó en Bolsa un 16% en dos días, lo que se tradujo en una pérdida de capitalización bursátil de 139.370 millones de dólares. Teniendo en cuenta que el magnate posee el 12,77% de las acciones de Tesla, unos 17.800 millones de dólares perdidos son suyos. En total, las acciones de Tesla se desploman más de un 35% en este 2025. "A esta caída 

In [5]:
import pandas as pd

# Definimos la batería de preguntas manualmente
datos = [
    {
        "id": 1,
        "tipo": "Factual",
        "pregunta": "¿A cuanto llego a subir Trump los aranceles a China?",
        "respuesta_esperada": "145%."
    },
    {
        "id": 2,
        "tipo": "Factual",
        "pregunta": "¿Cuánto dinero perdió Elon Musk tras el anuncio de los aranceles?",
        "respuesta_esperada": "17.800 millones de dólares en dos días."
    },
    
    {
        "id": 3,
        "tipo": "Trampa",
        "pregunta": "¿Qué opinó el Rey de España sobre los aranceles de Trump a China?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
     {
        "id": 4,
        "tipo": "Trampa",
        "pregunta": "¿Qué dijo Elon Musk sobre los aranceles a China?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    {
        "id": 5,
        "tipo": "Inferencia",
        "pregunta": "¿Qué relación quiere Elon Musk entre Estados Unidos y Europa según sus declaraciones en Italia?",
        "respuesta_esperada": "Quiere una zona de libre comercio con aranceles cero."
    },

    {
        "id": 6,
        "tipo": "Factual",
        "pregunta": "¿Qué presidente firmó un decreto para centralizar la regulación de la inteligencia artificial?",
        "respuesta_esperada": "Donald Trump."
    },

    {
        "id": 7,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué Donald Trump quiere quitar a los estados el derecho a regular la inteligencia artificial por su cuenta?",
        "respuesta_esperada": "Debe haber un único manual si queremos seguir liderando en inteligencia artificial"
    },

    {
        "id": 8,
        "tipo": "Trampa",
        "pregunta": "¿Qué opinó Felipe González sobre el decreto de inteligencia artificial de Trump?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    {
        "id": 9,
        "tipo": "Factual",
        "pregunta": "¿Por cuántas temporadas ha fichado Kylian Mbappé con el Real Madrid?",
        "respuesta_esperada": "Por cinco temporadas."
    },

    {
        "id": 10,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué el artículo de El País dice que el comunicado del fichaje de Mbappé cayó en medio de una resaca emocional?",
        "respuesta_esperada": "Porque el anuncio se hizo menos de 24 horas después de que terminaran las celebraciones por la 15ª Copa de Europa del Real Madrid."
    },

    {
        "id": 11,
        "tipo": "Factual",
        "pregunta": "¿Cuál dijo Pedro Sánchez que era su principal prioridad en la clausura del 41 Congreso Federal del PSOE?",
        "respuesta_esperada": "ganar las elecciones municipales y autonómicas de 2027 y volver a gobernar en toda España"
    },

    {
        "id": 12,
        "tipo": "Trampa", 
        "pregunta": "¿En qué fecha dimitió Pedro Sánchez?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    {
        "id": 13,
        "tipo": "Factual",
        "pregunta": "¿Cuanto patrimonio tiene Elon Musk?",
        "respuesta_esperada": "Mas de 400.000 millones de dólares"
    },
    {
        "id": 14,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué fue demandado Elon Musk por la Fiscalía de Filadelfia en 2024?",
        "respuesta_esperada": "Porque prometió dar 1 millón de dolares a votantes de las elecciones"
    },
    {
        "id": 15,
        "tipo": "Trampa", 
        "pregunta": "¿Cuántos bitcoins propuso repartir Donald Trump como renta básica universal?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 16,
        "tipo": "Factual", 
        "pregunta": "¿Cuántas personas murieron en la dana de Valencia?",
        "respuesta_esperada": "227 personas."
    },
    # --- GEOPOLÍTICA Y SOCIEDAD (Guerra en Ucrania) ---
    {
        "id": 17,
        "tipo": "Factual",
        "pregunta": "¿Qué enfermedades se asocian al uranio de los proyectiles antitanque utilizados en la guerra en Ucrania?",
        "respuesta_esperada": "Algunos estudios lo asocian con cáncer, problemas renales o efectos en el desarrollo infantil."
    },
    {
        "id": 18,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué el aumento de polacos y ucranianos que compran vivienda en España para huir de la guerra apenas recurre a la financiación de una hipoteca?",
        "respuesta_esperada": "Porque tienen un poder adquisitivo elevado y un patrimonio alto que les permite no necesitar una hipoteca."
    },
    {
        "id": 19,
        "tipo": "Factual",
        "pregunta": "¿A quién designó Donald Trump para servir como secretario de Defensa en 2025?",
        "respuesta_esperada": "A Pete Hegseth."
    },
    {
        "id": 20,
        "tipo": "Trampa",
        "pregunta": "¿Cuántos soldados estadounidenses ha prometido enviar Donald Trump para formar una fuerza de paz en Ucrania?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- DANA de Valencia ---
    {
        "id": 21,
        "tipo": "Factual",
        "pregunta": "¿Qué cantidad económica donó el Grupo Renault a los afectados por la DANA en Valencia y quién es el CEO de la compañía?",
        "respuesta_esperada": "Donó un millón de euros y su CEO es Luca de Meo."
    },
    {
        "id": 22,
        "tipo": "Inferencia",
        "pregunta": "Durante su concierto en Valencia, ¿qué canción improvisó Bryan Adams y con qué propósito en relación a la DANA?",
        "respuesta_esperada": "Improvisó una versión de 'Come Together' de The Beatles como símbolo de unión entre el artista y la comunidad afectada."
    },
    {
        "id": 23,
        "tipo": "Trampa",
        "pregunta": "¿Qué cantidad económica donó el futbolista Kylian Mbappé a los afectados por la DANA de Valencia?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 24,
        "tipo": "Inferencia",
        "pregunta": "¿A quien iba dirigido el mensaje en el tapiz floral de la Basílica durante la festividad de la patrona de Valencia tras el paso de la DANA?",
        "respuesta_esperada": "a los voluntarios que acudieron a las zonas afectadas para ayudar de manera desinteresada tras la dana"
    },
    {
        "id": 25,
        "tipo": "Factual",
        "pregunta": "¿Con cuántas psicólogas especialistas se reforzó la atención del teléfono 016 en los municipios afectados por la DANA en Valencia?",
        "respuesta_esperada": "Con cuatro psicólogas especialistas."
    },
    # --- ESPACIO Y TECNOLOGÍA (Misiones Lunares) ---
    {
        "id": 26,
        "tipo": "Factual",
        "pregunta": "Ante el retraso de SpaceX en la misión a la Luna y la actual guerra en la NASA por su jefatura, ¿a qué otra empresa mencionó el administrador interino Sean Duffy para abrir los contratos del módulo de aterrizaje?",
        "respuesta_esperada": "Mencionó a Blue Origin, la empresa de Jeff Bezos."
    },
    {
        "id": 27,
        "tipo": "Inferencia",
        "pregunta": "¿Qué fallo provocó que la nave Hakuto-R de SpaceX se estrellara en su intento anterior en 2023?",
        "respuesta_esperada": "Se estrelló porque calculó mal su descenso al pasar sobre un cráter, debido a datos imprecisos sobre la altitud y problemas de software."
    },
    {
        "id": 28,
        "tipo": "Trampa",
        "pregunta": "¿Qué astronauta lidera la tripulación del módulo Blue Ghost en las nuevas misiones a la Luna de SpaceX?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos (el texto especifica que es una misión no tripulada)."
    },

    # --- SALUD Y CIENCIA (Vacunas y Cáncer) ---
    {
        "id": 29,
        "tipo": "Inferencia",
        "pregunta": "¿Por qué la nueva vacuna de ARN para el tratamiento del cáncer se tiene que elaborar de manera personalizada para cada paciente?",
        "respuesta_esperada": "Porque los neoantígenos son proteínas anómalas que resultan de mutaciones genéticas, y el perfil de mutaciones es distinto y único en el tumor de cada paciente."
    },
    {
        "id": 30,
        "tipo": "Trampa",
        "pregunta": "¿Qué tipo de cáncer específico le fue diagnosticado al rey Carlos III?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- DEPORTES (Carlos Alcaraz) ---
    {
        "id": 31,
        "tipo": "Factual",
        "pregunta": "¿Cuánto dinero aproximado le quedó limpio a Carlos Alcaraz por ganar su sexto Grand Slam tras pagar los impuestos federales y de Nueva York?",
        "respuesta_esperada": "Le quedan aproximadamente 2,7 millones de dólares."
    },
    {
        "id": 32,
        "tipo": "Inferencia",
        "pregunta": "¿Qué golpe fundamental ha modificado Carlos Alcaraz en su juego para el Open de Australia??",
        "respuesta_esperada": "Ha modificado su saque (el servicio)."
    },

    # --- ECONOMÍA Y VIVIENDA ---
    {
        "id": 33,
        "tipo": "Factual",
        "pregunta": "Mientras el precio de la vivienda frenaba su subida en verano, ¿En que dos comunidades bajó el precio?",
        "respuesta_esperada": "En Castilla-La Mancha (caída del 12,8%) y Asturias (caída del 3,1%)."
    },
    {
        "id": 34,
        "tipo": "Inferencia",
        "pregunta": "¿Qué medida propone Santiago Niño Becerra para evitar la constante revalorización del mercado y poner solución a los problemas de vivienda en España?",
        "respuesta_esperada": "Propone crear un gran parque de vivienda pública destinado exclusivamente al alquiler."
    },

    # --- GEOPOLÍTICA TECNOLÓGICA (TikTok en EE.UU.) ---
    {
        "id": 35,
        "tipo": "Factual",
        "pregunta": "¿A partir de que día se prohibe Tiktok en Estados Unidos?",
        "respuesta_esperada": "A partir del domingo 19 de enero de 2025."
    },
    # --- DEPORTES (Eurocopa y Fútbol) ---
    {
        "id": 36,
        "tipo": "Factual",
        "pregunta": "¿Qué jugadores metieron gol para asegurar el 2-1 en el España-Inglaterra, dándole a España su cuarta Eurocopa?",
        "respuesta_esperada": "Nico Williams y Mikel Oyarzabal."
    },
    {
        "id": 37,
        "tipo": "Inferencia",
        "pregunta": "¿Qué dos hechos deportivos que sucedieron a la vez hicieron que L'Équipe titulara 'los nuevos reyes de España' señalando que el deporte español también tiene una fiesta nacional?",
        "respuesta_esperada": "Porque coincidió que en el mismo día los futbolistas de la selección se proclamaron campeones de Europa y el tenista Carlos Alcaraz retuvo su título en Wimbledon."
    },
    {
        "id": 38,
        "tipo": "Trampa",
        "pregunta": "Al ganar España la final de la Eurocopa 2024, ¿cuántos goles marcó el delantero inglés Harry Kane en su propia portería?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 39,
        "tipo": "Trampa",
        "pregunta": "Tras el partido en el que la España de las finales vuelve a ganar la Liga de Naciones, ¿qué entrenadora alemana fue destituida de su cargo?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- ECONOMÍA Y CONSUMO (Aceite de Oliva) ---
    {
        "id": 40,
        "tipo": "Factual",
        "pregunta": "Tras el giro del precio del aceite de oliva, ¿a cuánto vende el supermercado Alcampo su garrafa de 5 litros de la marca Suroliva?",
        "respuesta_esperada": "A 30,90 euros."
    },
    {
        "id": 41,
        "tipo": "Inferencia",
        "pregunta": "¿Cuáles son las razones por las que la garrafa de 5 litros de aceite Suroliva ha bajado a 30,90 euros?",
        "respuesta_esperada": "debido a una combinación de factores que incluyen la necesidad de liberar stock, la anticipación de una buena cosecha que estabilizará la oferta y la competencia entre supermercados para atraer a los consumidores."
    },
    {
        "id": 42,
        "tipo": "Trampa",
        "pregunta": "¿Qué cadena de supermercados está regalando garrafas de aceite de oliva en Madrid para compensar los problemas de la sequía?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- CULTURA Y SOCIEDAD (Taylor Swift) ---
    {
        "id": 43,
        "tipo": "Factual",
        "pregunta": "¿De cuántos millones ha sido el impacto económico en Madrid por la hostelería gracias a los conciertos de Taylor Swift?",
        "respuesta_esperada": "10 millones de euros."
    },
    {
        "id": 44,
        "tipo": "Inferencia",
        "pregunta": "Durante su enorme impacto económico en la gira musical, ¿por qué Taylor Swift decidió interpretar las reediciones denominadas 'Taylor's Version' de sus antiguos álbumes?",
        "respuesta_esperada": "Para recuperar la propiedad intelectual de su discografía después de que su discográfica anterior le quitara los derechos de sus primeros 7 albumes."
    },
    {
        "id": 45,
        "tipo": "Trampa",
        "pregunta": "Para los conciertos de Taylor Swift en Madrid, ¿qué famoso jugador de baloncesto subió al escenario del Santiago Bernabéu como invitado sorpresa?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 46,
        "tipo": "Trampa",
        "pregunta": "¿Cuánto dinero le cobró exactamente el presidente del Real Madrid a Taylor Swift por alquilar el estadio de fútbol?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- DEPORTES GLOBALES (Juegos Olímpicos) ---
    {
        "id": 47,
        "tipo": "Factual",
        "pregunta": "En relación con el calendario de los Juegos Olímpicos de París 2024, ¿en qué lugar exacto se celebrarán las competiciones de surf para aprovechar las olas?",
        "respuesta_esperada": "En Teahupo’o, Tahití."
    },
    {
        "id": 48,
        "tipo": "Trampa",
        "pregunta": "¿De qué material reciclado están fabricadas las tablas de surf con las que el equipo de España ganó las medallas en los Juegos Olímpicos de París 2024?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },

    # --- POLÍTICA INTERNACIONAL (Francia y Argentina) ---
    {
        "id": 49,
        "tipo": "Inferencia",
        "pregunta": "¿Qué decisión anunció Emmanuel Macron tras la victoria de Le Pen?",
        "respuesta_esperada": "Disolver de la Asamblea Nacional y convocar elecciones legislativas"
    },
    {
        "id": 50,
        "tipo": "Factual",
        "pregunta": "En los primeros cien días de su mandato para combatir la crisis, ¿a qué cifra de inflación anual se enfrentó Javier Milei en Argentina al cierre del año 2023?",
        "respuesta_esperada": "A una inflación anual del 211,4%."
    },
    {
        "id": 51,
        "tipo": "Inferencia",
        "pregunta": "Según la situación económica que vive la Argentina de Javier Milei con su alta inflación, ¿por qué motivo exacto una jueza de Nueva York rechazó al país la extensión del plazo para pagar la deuda de YPF?",
        "respuesta_esperada": "Porque consideró que el Gobierno argentino no había tomado medidas reales para afrontar el pago de la indemnización ni tenía un cronograma establecido para hacerlo."
    },
    {
        "id": 52,
        "tipo": "Trampa",
        "pregunta": "Tras convocar elecciones legislativas en Francia, ¿qué cargo público le ofreció Emmanuel Macron a Marine Le Pen para evitar dimitir?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 53,
        "tipo": "Trampa",
        "pregunta": "¿Con qué porcentaje exacto de votos ganó el partido de Los Republicanos las elecciones presidenciales de Francia frente a Macron?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 54,
        "tipo": "Trampa",
        "pregunta": "¿A qué jugador de la selección nacional de Francia despidió personalmente Javier Milei en Argentina tras el incidente con los cantos racistas?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos."
    },
    {
        "id": 55,
        "tipo": "Trampa",
        "pregunta": "¿Cuántos millones de dólares confesó haber robado Javier Milei en la estafa del llamado Criptogate en Argentina?",
        "respuesta_esperada": "No tengo información suficiente en mis archivos (el texto menciona acusaciones y demandas por promover $LIBRA, pero no una confesión de robo)."
    },
    {
        "id": 56,
        "tipo": "Factual",
        "pregunta": "¿Qué actor protagoniza la película 'On the Waterfront' (1953) mencionada en el artículo de los Óscar?",
        "respuesta_esperada": "Marlon Brando"
    },
    {
        "id": 57,
        "tipo": "Factual",
        "pregunta": "¿Qué sector profesional organizó las protestas conocidas como 'tractoradas' en distintos puntos de España?",
        "respuesta_esperada": "Los agricultores."
    },
    {
        "id": 58,
        "tipo": "Factual",
        "pregunta": "Según los reportes de RTVE de febrero de 2025, ¿con qué objetivo principal retomaron las tractoradas los agricultores en diferentes puntos de España?",
        "respuesta_esperada": "Para pedir cambios en el sector y exigir derechos."
    },
    {
        "id": 59,
        "tipo": "Factual",
        "pregunta": "¿En qué mes del año 2024 organizaron cientos de agricultores tractoradas con la intención de bloquear España?",
        "respuesta_esperada": "En el mes de febrero."
    },
    {
        "id": 60,
        "tipo": "Factual",
        "pregunta": "¿Qué tipo de vehículos utilizaron los agricultores en sus manifestaciones para intentar bloquear el país?",
        "respuesta_esperada": "Tractores."
    },
    {
        "id": 61,
        "tipo": "Inferencia",
        "pregunta": "¿qué paso en España con cientos de agricultores el martes 6 de febrero de 2024?",
        "respuesta_esperada": "El martes 6 de febrero de 2024, cientos de agricultores en España se movilizaron en protesta"
    },
    {
        "id": 62,
        "tipo": "Factual",
        "pregunta": "¿Qué película ganó el premio a Mejor Película en la edición de los premios Oscar de 2025?",
        "respuesta_esperada": "Anora."
    },
    {
        "id": 63,
        "tipo": "Factual",
        "pregunta": "¿Qué cineasta ganó el premio Oscar a Mejor Director en 2025 por la película 'Anora'?",
        "respuesta_esperada": "Sean Baker."
    },
    {
        "id": 64,
        "tipo": "Factual",
        "pregunta": "¿Qué actriz se llevó el Oscar a Mejor Actriz Protagonista por su papel en la película 'Anora'?",
        "respuesta_esperada": "Mikey Madison."
    },
    {
        "id": 65,
        "tipo": "Factual",
        "pregunta": "¿Qué actor logró el premio Oscar a Mejor Actor Protagonista en 2025 por su actuación en la película 'The Brutalist'?",
        "respuesta_esperada": "Adrien Brody."
    },
    {
        "id": 66,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue la película que ganó el Oscar a Mejor Película Internacional en 2025?",
        "respuesta_esperada": "I'm Still Here (Aún estoy aquí) de Brasil."
    },
    {
        "id": 67,
        "tipo": "Factual",
        "pregunta": "¿Qué partido político sufrió una derrota histórica en las elecciones del Reino Unido en julio de 2024?",
        "respuesta_esperada": "El Partido Conservador."
    },
    {
        "id": 68,
        "tipo": "Factual",
        "pregunta": "Tras los resultados de las elecciones en el Reino Unido de 2024, ¿qué partido político formó el nuevo Gobierno",
        "respuesta_esperada": "El Partido Laborista (Labour Party)."
    },
    {
        "id": 69,
        "tipo": "Factual",
        "pregunta": "¿En qué país europeo se celebraron elecciones a principios de julio de 2024 en las que los conservadores sufrieron una derrota histórica?",
        "respuesta_esperada": "En el Reino Unido."
    },
    {
        "id": 70,
        "tipo": "Factual",
        "pregunta": "¿Qué política se convirtió en la primera mujer en encabezar el Ministerio de Economía en la historia del Reino Unido al formarse el nuevo Gobierno en 2024?",
        "respuesta_esperada": "Rachel Reeves."
    },
    {
        "id": 71,
        "tipo": "Factual",
        "pregunta": "¿Quién se convirtió en el nuevo primer ministro del Reino Unido tras la victoria del Partido Laborista en julio de 2024?",
        "respuesta_esperada": "Keir Starmer."
    },
    {
        "id": 72,
        "tipo": "Factual",
        "pregunta": "¿Qué nombre reciben las esperadas gafas de realidad mixta lanzadas por la empresa Apple?",
        "respuesta_esperada": "Apple Vision Pro."
    },
    {
        "id": 73,
        "tipo": "Factual",
        "pregunta": "¿Cuál fue el precio oficial de salida al mercado de las gafas Apple Vision Pro en dólares?",
        "respuesta_esperada": "3.499 dólares."
    },
    {
        "id": 74,
        "tipo": "Inferencia",
        "pregunta": "¿Qué caracteristica tiene el nuevo modelo de Apple Vision Pro para poder conectarlas a los Mac?",
        "respuesta_esperada": "Una conexión física mediante cable."
    },
    {
        "id": 75,
        "tipo": "Factual",
        "pregunta": "¿En qué mes y año lanzó Apple al mercado estadounidense su primer dispositivo de gafas Vision Pro?",
        "respuesta_esperada": "En febrero de 2024."
    },
    {
        "id": 76,
        "tipo": "Factual",
        "pregunta": "¿Qué empresa tecnológica es la creadora y fabricante de las populares gafas de realidad mixta llamadas Vision Pro?",
        "respuesta_esperada": "Apple."
    },
    {
        "id": 77,
        "tipo": "Factual",
        "pregunta": "¿En qué fecha exacta se produjo el gran eclipse solar total que oscureció Norteamérica en 2024?",
        "respuesta_esperada": "El 8 de abril de 2024."
    },
    {
        "id": 78,
        "tipo": "Factual",
        "pregunta": "¿En qué tres países del continente americano se pudo observar la fase de totalidad del eclipse solar de abril de 2024?",
        "respuesta_esperada": "México, Estados Unidos y Canadá."
    },
    {
        "id": 79,
        "tipo": "Factual",
        "pregunta": "Según las noticias de abril de 2024, ¿qué fenómeno astronómico oscureció parte de Norteamérica atrayendo el gran interés de la ciencia?",
        "respuesta_esperada": "Un eclipse solar total."
    },
    {
        "id": 80,
        "tipo": "Inferencia",
        "pregunta": "¿En qué región concreta del continente americano se pudo aprovechar científicamente y de forma óptima el gran eclipse solar del 8 de abril de 2024?",
        "respuesta_esperada": "En América del Norte (Norteamérica)."
    }
    
]

# Convertimos a DataFrame
df_bateria = pd.DataFrame(datos)

# Mostramos una vista previa
print(f"✅ Batería creada con {len(df_bateria)} preguntas.")
display(df_bateria)

# Guardamos la batería en un archivo CSV para su uso posterior
df_bateria.to_csv("bateria_preguntas_tfg.csv", index=False)
print("✅ Batería guardada como 'bateria_preguntas_tfg.csv'.")

✅ Batería creada con 80 preguntas.


,id,tipo,pregunta,respuesta_esperada
0,1,Factual,¿A cuanto llego a subir Trump los aranceles a ...,145%.
1,2,Factual,¿Cuánto dinero perdió Elon Musk tras el anunci...,17.800 millones de dólares en dos días.
2,3,Trampa,¿Qué opinó el Rey de España sobre los arancele...,No tengo información suficiente en mis archivos.
3,4,Trampa,¿Qué dijo Elon Musk sobre los aranceles a China?,No tengo información suficiente en mis archivos.
4,5,Inferencia,¿Qué relación quiere Elon Musk entre Estados U...,Quiere una zona de libre comercio con arancele...
...,...,...,...,...
75,76,Factual,¿Qué empresa tecnológica es la creadora y fabr...,Apple.
76,77,Factual,¿En qué fecha exacta se produjo el gran eclips...,El 8 de abril de 2024.
77,78,Factual,¿En qué tres países del continente americano s...,"México, Estados Unidos y Canadá."
78,79,Factual,"Según las noticias de abril de 2024, ¿qué fenó...",Un eclipse solar total.


✅ Batería guardada como 'bateria_preguntas_tfg.csv'.
